# 🛡️ AI-Powered Scam Message Detector — Training Notebook

This notebook trains and compares three ML models to classify SMS/WhatsApp
messages as **SAFE**, **SUSPICIOUS**, or **SCAM**, then saves the best model
so it can be used in the Streamlit app.

**Pipeline:** Load data → TF-IDF vectorize → Train 3 models → Compare → Save best model


## 1. Install & import libraries

In [ ]:
!pip install -q scikit-learn pandas joblib

import json
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report


## 2. Load the dataset

Upload `scam_dataset.csv` to your Colab session (or mount Google Drive),
then load it below. The dataset has two columns: `message` and `label`
(SAFE / SUSPICIOUS / SCAM).

In [ ]:
from google.colab import files
uploaded = files.upload()  # select scam_dataset.csv when prompted


In [ ]:
df = pd.read_csv("scam_dataset.csv")
df = df.dropna(subset=["message", "label"])
print(f"Loaded {len(df)} messages")
df["label"].value_counts()


## 3. Train/test split

In [ ]:
X = df["message"].astype(str)
y = df["label"].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train size: {len(X_train)}  |  Test size: {len(X_test)}")


## 4. TF-IDF vectorization

In [ ]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_features=5000,
)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


## 5. Train & compare 3 candidate models

- **Logistic Regression** — fast, interpretable, strong text baseline
- **Multinomial Naive Bayes** — classic algorithm for spam/scam text classification
- **Linear SVM** — often the top performer on text classification tasks


In [ ]:
candidates = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Multinomial Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(class_weight="balanced", random_state=42, max_iter=5000),
}

results = {}
trained_models = {}

for name, model in candidates.items():
    model.fit(X_train_vec, y_train)
    preds = model.predict(X_test_vec)

    acc = accuracy_score(y_test, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, preds, average="weighted", zero_division=0)

    results[name] = {"accuracy": round(acc, 4), "precision": round(precision, 4),
                      "recall": round(recall, 4), "f1_score": round(f1, 4)}
    trained_models[name] = model

    print(f"\n{name}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  F1 Score : {f1:.4f}")
    print(classification_report(y_test, preds, zero_division=0))


## 6. Select the best model

In [ ]:
best_name = max(results, key=lambda k: results[k]["f1_score"])
best_model = trained_models[best_name]
print(f"Best model: {best_name}  (F1 = {results[best_name]['f1_score']})")

# LinearSVC has no predict_proba — calibrate it so the app can show confidence %
if best_name == "Linear SVM":
    final_model = CalibratedClassifierCV(best_model, cv=3)
    final_model.fit(X_train_vec, y_train)
else:
    final_model = best_model


## 7. Save model artifacts (download these 3 files for the Streamlit app)

In [ ]:
import os
os.makedirs("model", exist_ok=True)

joblib.dump(final_model, "model/model.pkl")
joblib.dump(vectorizer, "model/vectorizer.pkl")

with open("model/metrics.json", "w") as f:
    json.dump({
        "best_model": best_name,
        "results": results,
        "classes": sorted(y.unique().tolist()),
        "train_size": len(X_train),
        "test_size": len(X_test),
    }, f, indent=2)

print("Saved model.pkl, vectorizer.pkl, metrics.json")


In [ ]:
# Download the trained artifacts to your computer, then place them in the
# `model/` folder of your project before pushing to GitHub.
files.download("model/model.pkl")
files.download("model/vectorizer.pkl")
files.download("model/metrics.json")


## 8. Quick manual test

In [ ]:
def predict(text):
    vec = vectorizer.transform([text])
    label = final_model.predict(vec)[0]
    if hasattr(final_model, "predict_proba"):
        conf = max(final_model.predict_proba(vec)[0])
    else:
        conf = None
    return label, conf

for msg in [
    "Congratulations! You have WON $5000! Click bit.ly/claim-now to claim now.",
    "Hey, are we still meeting for lunch tomorrow?",
    "Reminder: your trial ends soon, upgrade to keep your benefits.",
]:
    label, conf = predict(msg)
    print(f"{label} ({conf*100:.1f}% confidence)  ->  {msg}")
